In [47]:
import requests
import json

In [54]:
url = "http://localhost:8001/v1/rerank"

In [75]:
instruction =  "Verify if the code provides a 'Puzzle Piece' for the specific target. Reject all code that does not explicitly use the target technology, even if it contains other critical vulnerabilities. Be a ruthless judge."
query = "H2Database: ALLOW_SYMLINKS and lack of path validation."
formatted_query = f"<Instruct>: {instruction}\n<Query>: {query}\n"

In [78]:
document = """
private static String encodeFileToBase64Binary(String fileName) throws IOException {
File file = new File(fileName);
byte[] encoded = Base64.encodeBase64(FileUtils.readFileToByteArray(file));
return new String(encoded, StandardCharsets.US_ASCII);
}"""

In [79]:
payload = {
    "model": "Qwen/Qwen3-Reranker-4B",
    "query": formatted_query,
    "documents": [document]
}

print("Pinging vLLM Server...")

try:
    response = requests.post(url, json=payload)
    response.raise_for_status()

    result = response.json()
    print("\n--- SERVER RESPONSE ---")
    print(json.dumps(result, indent=2))

    score = result['results'][0]['relevance_score']
    print("\n" + "="*50)
    print(f"FAISS Original Score: 0.5245")
    print(f"vLLM Reranker Score:  {score:.4f}")
    print("="*50 + "\n")

except Exception as e:
    print(f"Connection Failed: {e}")


--- SERVER RESPONSE ---
{
  "id": "rerank-b42f087b8d0b1ded",
  "model": "Qwen/Qwen3-Reranker-4B",
  "usage": {
    "prompt_tokens": 116,
    "total_tokens": 116
  },
  "results": [
    {
      "index": 0,
      "document": {
        "text": "\t\nprivate static String encodeFileToBase64Binary(String fileName) throws IOException {\nFile file = new File(fileName);\nbyte[] encoded = Base64.encodeBase64(FileUtils.readFileToByteArray(file));\nreturn new String(encoded, StandardCharsets.US_ASCII);\n}",
        "multi_modal": null
      },
      "relevance_score": 0.21544380486011505
    }
  ]
}

FAISS Original Score: 0.5245
vLLM Reranker Score:  0.2154

